In [ ]:
!pip install -q mediapipe==0.10.14 opencv-python-headless numpy pillow gdown

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 35.7/35.7 MB 29.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 294.9/294.9 kB 16.2 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
grpcio-status 1.71.2 requires protobuf<6.0dev,>=5.26.1, but you have protobuf 4.25.8 which is incompatible.
ydf 0.13.0 requires protobuf<7.0.0,>=5.29.1, but you have protobuf 4.25.8 which is incompatible.
opentelemetry-proto 1.37.0 requires protobuf<7.0,>=5.0, but you have protobuf 4.25.8 which is incompatible.


In [ ]:
import mediapipe as mp
print(f"✓ MediaPipe version: {mp.__version__}")

# Test if solutions work
mp_pose = mp.solutions.pose
mp_hands = mp.solutions.hands
mp_drawing = mp.solutions.drawing_utils
mp_drawing_styles = mp.solutions.drawing_styles

print("✓ All MediaPipe modules imported successfully!")

ModuleNotFoundError: No module named 'mediapipe'

In [ ]:
import cv2
import numpy as np
from datetime import timedelta
import gdown
import os
import json

print("✓ All libraries ready!")

✓ All libraries ready!


In [ ]:
file_id = '1FOvsbtPNvO_9MVwRS2S9WuWuldaFX8AF'

print("Downloading video...")

gdown.download(id=file_id, output='range-edited-test.mp4', quiet=False)

# Verify
if os.path.exists('range-edited-test.mp4'):
    size = os.path.getsize('range-edited-test.mp4') / (1024*1024)
    print(f"✓ Downloaded: {size:.1f} MB")
else:
    print("❌ Download failed!")

Downloading...
From (original): https://drive.google.com/uc?id=1FOvsbtPNvO_9MVwRS2S9WuWuldaFX8AF
From (redirected): https://drive.google.com/uc?id=1FOvsbtPNvO_9MVwRS2S9WuWuldaFX8AF&confirm=t&uuid=a8818a65-6e8a-40fc-9be3-75142c6f18ba
To: /content/range-edited-test.mp4
100%|██████████| 459M/459M [00:05<00:00, 80.5MB/s]

✓ Downloaded: 437.8 MB


In [ ]:
class VideoAnalyzer:
    def __init__(self):
        self.pose = mp_pose.Pose(
            static_image_mode=False,
            model_complexity=1,
            smooth_landmarks=True,
            min_detection_confidence=0.5,
            min_tracking_confidence=0.5
        )

        self.hands = mp_hands.Hands(
            static_image_mode=False,
            max_num_hands=2,
            min_detection_confidence=0.5,
            min_tracking_confidence=0.5
        )

        self.stats = {
            'total': 0,
            'pose': 0,
            'hands': 0,
            'weapon_events': []
        }

    def is_weapon_stance(self, pose_lm):
        if not pose_lm:
            return False

        try:
            lm = pose_lm.landmark
            lw = lm[mp_pose.PoseLandmark.LEFT_WRIST]
            rw = lm[mp_pose.PoseLandmark.RIGHT_WRIST]
            ls = lm[mp_pose.PoseLandmark.LEFT_SHOULDER]
            rs = lm[mp_pose.PoseLandmark.RIGHT_SHOULDER]
            lh = lm[mp_pose.PoseLandmark.LEFT_HIP]
            rh = lm[mp_pose.PoseLandmark.RIGHT_HIP]

            # Forward + Raised + Together
            forward = lw.z < ls.z - 0.1 or rw.z < rs.z - 0.1
            raised = lw.y < (lh.y + rh.y)/2 or rw.y < (lh.y + rh.y)/2
            together = abs(lw.x - rw.x) < 0.3

            return forward and raised and together
        except:
            return False

    def time_str(self, ms):
        td = timedelta(milliseconds=ms)
        s = int(td.total_seconds())
        return f"{s//60:02d}:{s%60:02d}:{td.microseconds//1000:03d}"

    def process(self, inp, out):
        cap = cv2.VideoCapture(inp)
        fps = int(cap.get(cv2.CAP_PROP_FPS))
        w = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
        h = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
        total = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))

        print(f"\n{w}x{h} @ {fps}fps")
        print(f"{total:,} frames ({total/fps/60:.1f} min)\n")

        writer = cv2.VideoWriter(out, cv2.VideoWriter_fourcc(*'mp4v'), fps, (w,h))

        n = 0
        while True:
            ret, frame = cap.read()
            if not ret:
                break

            n += 1
            self.stats['total'] = n

            if n % 100 == 0:
                print(f"{int(n/total*100)}% ", end='', flush=True)

            rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)

            # POSE (always)
            pose_res = self.pose.process(rgb)
            weapon = False

            if pose_res.pose_landmarks:
                self.stats['pose'] += 1
                weapon = self.is_weapon_stance(pose_res.pose_landmarks)

                mp_drawing.draw_landmarks(
                    frame, pose_res.pose_landmarks, mp_pose.POSE_CONNECTIONS,
                    mp_drawing_styles.get_default_pose_landmarks_style()
                )

            # HANDS (conditional)
            if weapon:
                hand_res = self.hands.process(rgb)
                if hand_res.multi_hand_landmarks:
                    self.stats['hands'] += 1
                    self.stats['weapon_events'].append({
                        'frame': n,
                        'time': self.time_str(int(cap.get(cv2.CAP_PROP_POS_MSEC)))
                    })

                    for hand in hand_res.multi_hand_landmarks:
                        mp_drawing.draw_landmarks(
                            frame, hand, mp_hands.HAND_CONNECTIONS,
                            mp_drawing_styles.get_default_hand_landmarks_style(),
                            mp_drawing_styles.get_default_hand_connections_style()
                        )

            # Overlays
            ms = int(cap.get(cv2.CAP_PROP_POS_MSEC))
            cv2.putText(frame, self.time_str(ms), (w-200,50),
                       cv2.FONT_HERSHEY_SIMPLEX, 1, (0,255,0), 2)

            status = "POSE+HANDS" if weapon else "POSE"
            color = (0,255,0) if weapon else (255,255,0)
            cv2.putText(frame, status, (10,50),
                       cv2.FONT_HERSHEY_SIMPLEX, 1, color, 2)

            writer.write(frame)

        cap.release()
        writer.release()

        print("\n\n✓ DONE!\n")
        print(f"Frames: {self.stats['total']:,}")
        print(f"Pose: {self.stats['pose']:,} ({self.stats['pose']/self.stats['total']*100:.1f}%)")
        print(f"Hands: {self.stats['hands']:,}")
        print(f"Events: {len(self.stats['weapon_events'])}\n")

        if self.stats['weapon_events']:
            print("First 5 events:")
            for i, e in enumerate(self.stats['weapon_events'][:5], 1):
                print(f"  {i}. Frame {e['frame']:,} @ {e['time']}")

        return self.stats

    def close(self):
        self.pose.close()
        self.hands.close()

print("✓ Analyzer ready")

✓ Analyzer ready


In [ ]:
analyzer = VideoAnalyzer()

try:
    results = analyzer.process('/content/range-edited-test.mp4', 'output.mp4')

    with open('results.json', 'w') as f:
        json.dump(results, f, indent=2)

    print("\n✓ Saved: output.mp4, results.json")

except Exception as e:
    print(f"\n❌ Error: {e}")
    import traceback
    traceback.print_exc()

finally:
    analyzer.close()


3640x2048 @ 29fps
29,278 frames (16.8 min)



/usr/local/lib/python3.12/dist-packages/google/protobuf/symbol_database.py:55: UserWarning: SymbolDatabase.GetPrototype() is deprecated. Please use message_factory.GetMessageClass() instead. SymbolDatabase.GetPrototype() will be removed soon.
  warnings.warn('SymbolDatabase.GetPrototype() is deprecated. Please '


0% 0% 1% 1% 1% 2% 2% 2% 3% 3% 3% 4% 4% 4% 5% 5% 5% 6% 6% 6% 7% 7% 7% 8% 8% 8% 9% 9% 9% 10% 10% 10% 11% 11% 11% 12% 12% 12% 13% 13% 14% 14% 14% 15% 15% 15% 16% 16% 16% 17% 17% 17% 18% 18% 18% 19% 19% 19% 20% 20% 20% 21% 21% 21% 22% 22% 22% 23% 23% 23% 24% 24% 24% 25% 25% 25% 26% 26% 26% 27% 27% 28% 28% 28% 29% 29% 29% 30% 30% 30% 31% 31% 31% 32% 32% 32% 33% 33% 33% 34% 34% 34% 35% 35% 35% 36% 36% 36% 37% 37% 37% 38% 38% 38% 39% 39% 39% 40% 40% 40% 41% 41% 42% 42% 42% 43% 43% 43% 44% 44% 44% 45% 45% 45% 46% 46% 46% 47% 47% 47% 48% 48% 48% 49% 49% 49% 50% 50% 50% 51% 51% 51% 52% 52% 52% 53% 53% 53% 54% 54% 54% 55% 55% 56% 56% 56% 57% 57% 57% 58% 58% 58% 59% 59% 59% 60% 60% 60% 61% 61% 61% 62% 62% 62% 63% 63% 63% 64% 64% 64% 65% 65% 65% 66% 